# 2) Adapter Egitimi (Colab)

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [13]:
#@title Secrets
from google.colab import userdata

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print('[GIT] GitHub token Colab secret/env uzerinden bulundu.')
else:
    print('[GIT] GitHub token bulunamadi. Public read disinda clone/push gerekiyorsa GH_TOKEN ekleyin.')

if hf_token:
    print('[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.')
else:
    print('[HF] Hugging Face token bulunamadi. Gated model gerekiyorsa Colab secret ekleyin.')

CLONE_TARGET = Path('/content/bitirmeprojesi')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _is_repo_root_inline(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _build_repo_access_url(repo_url: str, token: str | None) -> str:
    _ = token  # token URL'ye yazilmaz; log/telemetry sizintisi onlenir.
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return str(repo_url or '').strip()
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, netloc, parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root_inline(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root_inline(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root_inline(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root_inline(child):
                return child
    return None

def _ensure_repo_root_for_update_check() -> Path | None:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, gh_token)
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    return _find_repo_root_inline()

repo_root_for_update_check = _ensure_repo_root_for_update_check()
if repo_root_for_update_check is not None:
    if str(repo_root_for_update_check) not in sys.path:
        sys.path.insert(0, str(repo_root_for_update_check))
    try:
        from scripts.colab_repo_bootstrap import probe_repo_update_status

        update_status = probe_repo_update_status(repo_root_for_update_check)
        relation = str(update_status.get('relation', 'unknown'))
        if relation == 'up_to_date':
            print('[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.')
        elif relation == 'update_available':
            print(f"[KONTROL] Ilk hucre: guncelleme var. Branch={update_status.get('branch', '')}")
        else:
            print(f"[KONTROL] Ilk hucre: guncelleme durumu okunamadi: {update_status.get('message', 'bilgi yok')}")
    except Exception as exc:
        print(f'[KONTROL] Ilk hucrede guncellik kontrolu basarisiz: {exc}')
else:
    print('[KONTROL] Ilk hucrede repo bulunamadi ve auto-clone da basarisiz oldu.')


[GIT] GitHub token Colab secret/env uzerinden bulundu.
[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.
[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.


In [14]:
#@title Run Identity
# Notebook 2 calisma kimligi
# Once bu hucreyi duzenleyin, sonra bootstrap hucresini yeniden calistirin.
CROP_NAME = "grape"
PART_NAME = "fruit"
print(f"[RUN] crop={CROP_NAME} part={PART_NAME}")


[RUN] crop=grape part=fruit


In [15]:
#@title Bootstrap
import io
import json
import os
import random
import shutil
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: repo kokunu bul, bagimliliklari kur ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    _ = token  # token URL'ye yazilmaz; log/telemetry sizintisi onlenir.
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return str(repo_url or '').strip()
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, netloc, parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager
from scripts.colab_notebook_helpers import build_notebook_run_id


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
CROP_NAME = globals().get('CROP_NAME', 'tomato')
PART_NAME = globals().get('PART_NAME', 'unspecified')
RUN_ID = build_notebook_run_id(CROP_NAME, PART_NAME)
NOTEBOOK_FILENAME = '2_interactive_adapter_training.executed.ipynb'
REPO_RUN_DIR = ROOT / 'runs' / RUN_ID
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_training'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_training'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

LOCAL_TELEMETRY_ROOT = ROOT / 'outputs' / 'colab_notebook_training' / 'telemetry_runtime'
LOCAL_TELEMETRY_SPOOL_ROOT = ROOT / '.runtime_tmp' / 'colab_notebook_training' / 'telemetry_spool'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='2_interactive_adapter_training.ipynb',
    run_id=RUN_ID,
    drive_root=LOCAL_TELEMETRY_ROOT,
    local_root=LOCAL_TELEMETRY_SPOOL_ROOT,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR, exclude_dir_names=("checkpoints", "telemetry_runtime"))
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

TELEMETRY.configure_repo_output_export(
    notebook_path=REPO_NOTEBOOK_OUTPUT_PATH,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} crop={CROP_NAME} part={PART_NAME} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


[BOOTSTRAP] run_id=grape_fruit_2026-04-14_10-56-34 crop=grape part=fruit capture_cell_output_compat=False


In [16]:
#@title Training Parameters
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri ---
    # Bu hucreyi duzenleyin, sonra kalan hucreleri sirayla calistirin.
    # Kosu kimligi icin CROP_NAME/PART_NAME degerlerini ustteki hucreden yonetin.

    # RUNTIME_DATASET_ROOT: Notebook 0'un yazdigi <dataset_key>/continual|val|test|ood yapisini tutan repo-ici root.
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets/grape__fruit"

    # DATASET_NAME: Notebook 0'un urettigi runtime dataset klasor adi. Bos ise notebook kullaniciya sorar.
    DATASET_NAME = ""

    # CROP_NAME ve PART_NAME, kosu adlandirmasi ve metadata icin kullanilir.
    CROP_NAME = globals().get("CROP_NAME", "tomato")
    PART_NAME = globals().get("PART_NAME", "unspecified")

    # FEW_SHOT_RESEARCH_MODE: True olursa 100 image/class production guardrail'i research kosulari icin bypass edilir.
    FEW_SHOT_RESEARCH_MODE = False
    FEW_SHOT_MIN_CLASS_SAMPLES = 1

    # EPOCHS: train split uzerinden kac tam gecis yapilacagi.
    EPOCHS = 20

    # BATCH_SIZE: optimizer adimi basina ornek sayisi. GPU limitine yakin olacak sekilde artirilabilir.
    BATCH_SIZE = 128

    # LEARNING_RATE: adapter/LoRA parametreleri icin optimizer adim buyuklugu.
    LEARNING_RATE = 2e-4

    # LORA_R: LoRA rank degeri. Buyudukce kapasite ve VRAM/islem maliyeti artar.
    LORA_R = 24

    # LORA_ALPHA: LoRA olcekleme katsayisi. Genelde LORA_R degerinin 2x-4x araliginda kullanilir.
    LORA_ALPHA = 24

    # LORA_DROPOUT: LoRA katmanlarina uygulanan dropout. Buyudukce regularizasyon artar.
    LORA_DROPOUT = 0.1

    # OOD_FACTOR: OOD esik hassasiyetini carpansal olarak ayarlar.
    OOD_FACTOR = 3.0
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: eski/yeni sinif enerji ayrimi icin deneysel egitim regularizeri.
    BER_ENABLED = False

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: eski ve yeni sinif kisimlari icin BER ceza agirliklari.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW weight decay degeri.
    WEIGHT_DECAY = 0.01

    # MIXED_PRECISION: {'off', 'auto', 'fp16', 'bf16'} seceneklerinden biri.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation katsayisi.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping esigi. 0 olursa clipping kapanir.
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing katsayisi.
    LABEL_SMOOTHING = 0.0

    # LOSS_NAME / LOGITNORM_TAU: varsayilan kayip LogitNorm'dur; CE icin loss_name='cross_entropy' secin.
    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    # Scheduler ayarlari.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping ayarlari.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    # Tekrarlanabilirlik ve runtime ayarlari.
    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    # NUM_WORKERS: dataloader worker sayisi. CPU veri yukleme paralelligini belirler.
    NUM_WORKERS = 12

    # PREFETCH: NUM_WORKERS > 0 iken worker basina prefetch katsayisi.
    PREFETCH = 8

    # PIN_MEMORY: host memory sabitleyerek host-to-GPU aktarimini hizlandirir.
    PIN_MEMORY = True

    # USE_CACHE: destekleniyorsa decode edilmis ornekleri RAM'de tutar.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: continual/train split'ini de cache'ler. Yuksek RAM'li Colab icin uygundur.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: her N epoch'ta tam validation calistirir; final epoch her zaman dahildir.
    VALIDATION_EVERY_N_EPOCHS = 2

    # CHECKPOINT_EVERY_N_STEPS / CHECKPOINT_ON_EXCEPTION: notebook checkpoint sikligi ayarlari.
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True

    # STDOUT_BATCH_INTERVAL: canli training ilerleme yazdirma araligi.
    STDOUT_BATCH_INTERVAL = 12

    # RESUME_MODE: "fresh" yeni kosu baslatir, "resume" son checkpointten devam eder.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # OPTIMIZATION_CAMPAIGN_MODE: "disabled" tek kosu, "continue" ayni dataset kohortu icin optimizer kampanyasina devam,
    # "stop" ise mevcut kampanyayi durdurur ve yeni onerileri uygulamaz.
    OPTIMIZATION_CAMPAIGN_MODE = "continue"  # "disabled", "continue", "stop"

    # OPTIMIZATION_MAX_SESSION_RUNS: notebook gorunurlugu icin her calistirmada uygulanacak optimizer adimi sayisi.
    # Simdilik Notebook 2 bir oturumda 1 oneriyi uygular; devam etmek icin notebook yeniden calistirilir.
    OPTIMIZATION_MAX_SESSION_RUNS = 3

    # AUTO_DISCONNECT_RUNTIME: tum final exportlar basariliysa Colab runtime'i kapatir.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: son durum gorunsun diye disconnect oncesi kisa bekleme suresi.
    AUTO_DISCONNECT_GRACE_SECONDS = 20

    # AUTO_PUSH_TO_GITHUB: final exportlar bitince runs/<RUN_ID>/ klasorunu repoya commit edip pushlar.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: auto-push aciksa kullanilacak git remote adi.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: auto-push icin branch override degeri. None olursa mevcut branch kullanilir.
    AUTO_PUSH_BRANCH = None

    # Notebook icinde daha sonra kullanilacak gizli repo varsayimlarini yukle.
    BASE_CONFIG = ConfigurationManager(config_dir=str(ROOT / "config"), environment="colab").load_all_configs()
    CONTINUAL_DATA_CFG = BASE_CONFIG.get("training", {}).get("continual", {}).get("data", {})

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = bool(TF32_ENABLED)

    TARGET_SIZE = int(CONTINUAL_DATA_CFG.get("target_size", 224))
    DATA_SAMPLER = str(CONTINUAL_DATA_CFG.get("sampler", "shuffle"))
    AUGMENTATION_POLICY = str(CONTINUAL_DATA_CFG.get("augmentation_policy", "randaugment")).strip().lower()
    RANDAUGMENT_NUM_OPS = int(CONTINUAL_DATA_CFG.get("randaugment_num_ops", 2))
    RANDAUGMENT_MAGNITUDE = int(CONTINUAL_DATA_CFG.get("randaugment_magnitude", 7))
    FEW_SHOT_RESEARCH_MODE = bool(FEW_SHOT_RESEARCH_MODE)
    FEW_SHOT_MIN_CLASS_SAMPLES = int(FEW_SHOT_MIN_CLASS_SAMPLES)
    LOADER_ERROR_POLICY = str(CONTINUAL_DATA_CFG.get("loader_error_policy", "tolerant"))
    CACHE_SIZE = int(CONTINUAL_DATA_CFG.get("cache_size", 1000))
    VALIDATE_IMAGES_ON_INIT = bool(CONTINUAL_DATA_CFG.get("validate_images_on_init", True))

    STATE = {
        "validated": False,
        "class_names": [],
        "runtime_dataset_root": None,
        "runtime_dataset_key": None,
        "selected_dataset_name": None,
        "selected_dataset_root": None,
        "resolved_ood_root": None,
        "adapter": None,
        "loaders": None,
        "history": None,
        "calibration": None,
        "checkpoint_manager": CHECKPOINT_MANAGER,
        "resume_manifest": None,
        "best_val_loss": None,
        "best_metric_state": {},
        "optimization_campaign": None,
        "optimization_applied_params": None,
        "auto_disconnect_report": None,
        "git_push_report": None,
        "telemetry": TELEMETRY,
    }

    if RESUME_MODE == "resume":
        try:
            STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
            if STATE["resume_manifest"]:
                manifest = STATE["resume_manifest"]
                print(
                    f"[RESUME] Checkpoint bulundu: {manifest.get('name', '?')} "
                    f"epoch={manifest.get('epoch', 0)} step={manifest.get('global_step', 0)}"
                )
        except Exception:
            pass

    hf_token_ready = False
    hf_token = str(resolve_hf_token() or "").strip()
    if not hf_token:
        raise RuntimeError(
            "Notebook 2 egitim baslamadan once Hugging Face token ister. "
            "HF_TOKEN, HUGGINGFACE_TOKEN veya HUGGINGFACE_HUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
        )
    hf_token_ready = bool(login_and_check_hf_token(print_fn=print))
    if not hf_token_ready:
        raise RuntimeError(
            "Hugging Face token on kontrolu basarisiz oldu. Egitimden once tokeni duzeltin."
        )

    github_token_ready = False
    if AUTO_PUSH_TO_GITHUB:
        github_token = str(resolve_github_token() or "").strip()
        if not github_token:
            raise RuntimeError(
                "AUTO_PUSH_TO_GITHUB is enabled, but no GitHub token was found. "
                "Egitim baslamadan once GH_TOKEN veya GITHUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
            )
        github_token_ready = True
        print("[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.")

    print(
        f"[CONFIG] source=notebook_cell crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} "
        f"lr={LEARNING_RATE} resume={RESUME_MODE} ber={BER_ENABLED} "
        f"loss={LOSS_NAME} tau={LOGITNORM_TAU} opt_mode={OPTIMIZATION_CAMPAIGN_MODE} "
        f"opt_session_limit={OPTIMIZATION_MAX_SESSION_RUNS} auto_disconnect={AUTO_DISCONNECT_RUNTIME} "
        f"auto_push={AUTO_PUSH_TO_GITHUB}"
    )
    print(
        f"[DATASET] runtime_root={RUNTIME_DATASET_ROOT} dataset_name={DATASET_NAME or '<ask>'}"
    )
    print(
        "[OOD] Notebook 2 harici OOD secmez; gercek OOD kaniti icin secilen runtime dataset icinde ood/ klasoru bulunmalidir."
    )
    print(
        f"[RUNTIME] defaults=notebook_cell mp={MIXED_PRECISION} workers={NUM_WORKERS} prefetch={PREFETCH} "
        f"sched={SCHEDULER_NAME} wd={WEIGHT_DECAY} accum={GRAD_ACCUM_STEPS} grad_clip={MAX_GRAD_NORM} "
        f"label_smooth={LABEL_SMOOTHING} warmup={SCHEDULER_WARMUP_RATIO} "
        f"early_stop={EARLY_STOPPING_PATIENCE}/{EARLY_STOPPING_MIN_DELTA} "
        f"val_every={VALIDATION_EVERY_N_EPOCHS} cache_train={CACHE_TRAIN_SPLIT} "
        f"aug={AUGMENTATION_POLICY} randaug={RANDAUGMENT_NUM_OPS}/{RANDAUGMENT_MAGNITUDE} "
        f"few_shot_research={FEW_SHOT_RESEARCH_MODE} min_class={FEW_SHOT_MIN_CLASS_SAMPLES}"
    )
    print(
        f"[OOD] factor={OOD_FACTOR} sure={SURE_SEMANTIC_PERCENTILE}/{SURE_CONFIDENCE_PERCENTILE} "
        f"conformal={CONFORMAL_METHOD} alpha={CONFORMAL_ALPHA} "
        f"raps_lambda={CONFORMAL_RAPS_LAMBDA} raps_k={CONFORMAL_RAPS_K_REG}"
    )
    TELEMETRY.update_latest(
        {
            "phase": "parameters_ready",
            "parameter_source": "notebook_cell",
            "crop": CROP_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "dataset_name": DATASET_NAME,
            "loss_name": LOSS_NAME,
            "logitnorm_tau": LOGITNORM_TAU,
            "mixed_precision": MIXED_PRECISION,
            "num_workers": NUM_WORKERS,
            "prefetch": PREFETCH,
            "cache_train_split": CACHE_TRAIN_SPLIT,
            "augmentation_policy": AUGMENTATION_POLICY,
            "randaugment_num_ops": RANDAUGMENT_NUM_OPS,
            "randaugment_magnitude": RANDAUGMENT_MAGNITUDE,
            "few_shot_research_mode": FEW_SHOT_RESEARCH_MODE,
            "few_shot_min_class_samples": FEW_SHOT_MIN_CLASS_SAMPLES,
            "ber_enabled": BER_ENABLED,
            "ber_lambda_old": BER_LAMBDA_OLD,
            "ber_lambda_new": BER_LAMBDA_NEW,
            "ber_warmup_steps": BER_WARMUP_STEPS,
            "ood_factor": OOD_FACTOR,
            "sure_semantic_percentile": SURE_SEMANTIC_PERCENTILE,
            "sure_confidence_percentile": SURE_CONFIDENCE_PERCENTILE,
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_method": CONFORMAL_METHOD,
            "conformal_raps_lambda": CONFORMAL_RAPS_LAMBDA,
            "conformal_raps_k_reg": CONFORMAL_RAPS_K_REG,
            "label_smoothing": LABEL_SMOOTHING,
            "scheduler_warmup_ratio": SCHEDULER_WARMUP_RATIO,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "resume_mode": RESUME_MODE,
            "optimization_campaign_mode": OPTIMIZATION_CAMPAIGN_MODE,
            "optimization_max_session_runs": OPTIMIZATION_MAX_SESSION_RUNS,
            "validation_every_n_epochs": VALIDATION_EVERY_N_EPOCHS,
            "auto_push_to_github": AUTO_PUSH_TO_GITHUB,
            "hf_token_ready": hf_token_ready,
            "auto_push_token_ready": github_token_ready,
            "auto_push_remote_name": AUTO_PUSH_REMOTE_NAME,
            "auto_push_branch": AUTO_PUSH_BRANCH,
            "auto_disconnect_runtime": AUTO_DISCONNECT_RUNTIME,
            "auto_disconnect_grace_seconds": AUTO_DISCONNECT_GRACE_SECONDS,
        }
    )


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=grape epochs=20 bs=128 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 opt_mode=continue opt_session_limit=3 auto_disconnect=True auto_push=True
[DATASET] runtime_root=data/prepared_runtime_datasets/grape__fruit dataset_name=<ask>
[OOD] Notebook 2 harici OOD secmez; gercek OOD kaniti icin secilen runtime dataset icinde ood/ klasoru bulunmalidir.
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=2 cache_train=True aug=randaugment randaug=2/7 few_shot_research=False min_class=1
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [17]:
#@title Environment + Dataset Prep
with TELEMETRY.capture_cell_output("Cell 3b: Guncelleme ve Erisim Kontrolu"):
    # Access checks are delegated to prepare_notebook_access_and_dataset via
    # collect_notebook_access_report and print_notebook_access_report.
    from scripts.colab_notebook_helpers import prepare_notebook_access_and_dataset

    preparation = prepare_notebook_access_and_dataset(
        root=ROOT,
        base_config=BASE_CONFIG,
        crop_name=CROP_NAME,
        dataset_name=DATASET_NAME,
        runtime_dataset_root=RUNTIME_DATASET_ROOT,
        optimization_campaign_mode=OPTIMIZATION_CAMPAIGN_MODE,
        telemetry=TELEMETRY,
        print_fn=print,
    )
    STATE.update(preparation)


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.
[DATASET] Yalnizca bir runtime dataset bulundu, otomatik secildi: grape__fruit
[OOD] runtime ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__fruit/ood
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__fruit classes=5: ['anthracnose', 'botrytis_bunch_rot', 'downy_mildew', 'healthy', 'powdery_mildew']
[OPT] mode=continue dataset_lineage will be resolved from /content/bitirmeprojesi/data/prepared_runtime_datasets/grape__fruit/split_manifest.json
[OPT] Detailed cohort status and the next proposal will be resolved during engine init.


In [18]:
#@title Engine Init
with TELEMETRY.capture_cell_output("Cell 4: Engine Init"):
    from scripts.colab_notebook_helpers import initialize_notebook_training_engine

    engine_result = initialize_notebook_training_engine(
        root=ROOT,
        state=STATE,
        base_config=BASE_CONFIG,
        crop_name=CROP_NAME,
        part_name=PART_NAME,
        device=DEVICE,
        deterministic=DETERMINISTIC,
        notebook_parameters={
            "EPOCHS": EPOCHS,
            "BATCH_SIZE": BATCH_SIZE,
            "LEARNING_RATE": LEARNING_RATE,
            "LORA_R": LORA_R,
            "LORA_ALPHA": LORA_ALPHA,
            "LORA_DROPOUT": LORA_DROPOUT,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "OOD_FACTOR": OOD_FACTOR,
            "LOGITNORM_TAU": LOGITNORM_TAU,
            "RANDAUGMENT_MAGNITUDE": RANDAUGMENT_MAGNITUDE,
        },
        optimization_campaign_mode=OPTIMIZATION_CAMPAIGN_MODE,
        data_settings={
            "AUGMENTATION_POLICY": AUGMENTATION_POLICY,
            "RANDAUGMENT_NUM_OPS": RANDAUGMENT_NUM_OPS,
            "FEW_SHOT_RESEARCH_MODE": FEW_SHOT_RESEARCH_MODE,
            "FEW_SHOT_MIN_CLASS_SAMPLES": FEW_SHOT_MIN_CLASS_SAMPLES,
        },
        loader_settings={
            "NUM_WORKERS": NUM_WORKERS,
            "PREFETCH": PREFETCH,
            "USE_CACHE": USE_CACHE,
            "CACHE_SIZE": CACHE_SIZE,
            "CACHE_TRAIN_SPLIT": CACHE_TRAIN_SPLIT,
            "TARGET_SIZE": TARGET_SIZE,
            "LOADER_ERROR_POLICY": LOADER_ERROR_POLICY,
            "DATA_SAMPLER": DATA_SAMPLER,
            "SEED": SEED,
            "VALIDATE_IMAGES_ON_INIT": VALIDATE_IMAGES_ON_INIT,
            "PIN_MEMORY": PIN_MEMORY,
        },
        ood_settings={
            "sure_semantic_percentile": SURE_SEMANTIC_PERCENTILE,
            "sure_confidence_percentile": SURE_CONFIDENCE_PERCENTILE,
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_method": CONFORMAL_METHOD,
            "conformal_raps_lambda": CONFORMAL_RAPS_LAMBDA,
            "conformal_raps_k_reg": CONFORMAL_RAPS_K_REG,
            "ber_enabled": BER_ENABLED,
            "ber_lambda_old": BER_LAMBDA_OLD,
            "ber_lambda_new": BER_LAMBDA_NEW,
            "ber_warmup_steps": BER_WARMUP_STEPS,
        },
        optimization_settings={
            "grad_accumulation_steps": GRAD_ACCUM_STEPS,
            "max_grad_norm": MAX_GRAD_NORM,
            "mixed_precision": MIXED_PRECISION,
            "label_smoothing": LABEL_SMOOTHING,
            "loss_name": LOSS_NAME,
            "scheduler": {
                "name": SCHEDULER_NAME,
                "warmup_ratio": SCHEDULER_WARMUP_RATIO,
                "min_lr": SCHEDULER_MIN_LR,
                "step_on": "batch",
            },
        },
        early_stopping_settings={
            "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
            "EARLY_STOPPING_MIN_DELTA": EARLY_STOPPING_MIN_DELTA,
        },
        telemetry=TELEMETRY,
        print_fn=print,
    )
    resolved_notebook_params = engine_result["notebook_parameters"]
    EPOCHS = int(resolved_notebook_params["EPOCHS"])
    BATCH_SIZE = int(resolved_notebook_params["BATCH_SIZE"])
    LEARNING_RATE = float(resolved_notebook_params["LEARNING_RATE"])
    LORA_R = int(resolved_notebook_params["LORA_R"])
    LORA_ALPHA = int(resolved_notebook_params["LORA_ALPHA"])
    LORA_DROPOUT = float(resolved_notebook_params["LORA_DROPOUT"])
    WEIGHT_DECAY = float(resolved_notebook_params["WEIGHT_DECAY"])
    OOD_FACTOR = float(resolved_notebook_params["OOD_FACTOR"])
    LOGITNORM_TAU = float(resolved_notebook_params["LOGITNORM_TAU"])
    RANDAUGMENT_MAGNITUDE = int(resolved_notebook_params["RANDAUGMENT_MAGNITUDE"])


[OOD] runtime ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__fruit/ood
[CLASSES] ['anthracnose', 'botrytis_bunch_rot', 'downy_mildew', 'healthy', 'powdery_mildew']
[CLASSES] mode=taxonomy_filter reason=full_taxonomy_alignment matched=5 unmatched=0
[OPT] mode=continue status=bootstrap_pending eligible=0 frontier=0 executed=1
[OPT] No prior comparable run yet. Bootstrap run will use the visible notebook parameters.
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature": 1.0, "energy_temperature_mode": "fixed", "energy_temperature_range": [0.5, 3.0], "energy_temperature_steps": 16, "knn_backend": "auto", "knn_chunk_size": 2048, "primary_score_method": "auto", "radial_beta_range": [0.5, 2.0], "radial_beta_steps": 16, "radial_l2_enabled": true, "sure_confid

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.
[ENGINE] Hazir. trainable_params=13,771,017  classes=5


In [19]:
#@title Training
# run_id = RUN_ID
with TELEMETRY.capture_cell_output("Cell 5: Training"):
    from scripts.colab_notebook_helpers import run_notebook_training_session

    training_result = run_notebook_training_session(
        root=ROOT,
        state=STATE,
        run_id=RUN_ID,
        epochs=EPOCHS,
        device=DEVICE,
        stdout_batch_interval=STDOUT_BATCH_INTERVAL,
        validation_every_n_epochs=VALIDATION_EVERY_N_EPOCHS,
        checkpoint_every_n_steps=CHECKPOINT_EVERY_N_STEPS,
        checkpoint_on_exception=CHECKPOINT_ON_EXCEPTION,
        resume_mode=RESUME_MODE,
        telemetry=TELEMETRY,
        print_fn=print,
    )
    STATE.update(training_result["state"])


[TRAIN] epochs=20 device=cuda batch_interval=12
[LIVE] 1/20 batch=1/6 loss=1.7063 lr=0.000200 throughput=263.3/s elapsed=3s eta=5m31s
[LIVE] 1/20 batch=6/6 loss=1.6225 lr=0.000199 throughput=274.3/s elapsed=5s eta=1m38s
[VALID] 1/20


/content/bitirmeprojesi/scripts/colab_notebook_helpers.py:2089: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
/content/bitirmeprojesi/scripts/colab_notebook_helpers.py:2099: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


[LIVE] 2/20 batch=1/6 loss=1.6090 lr=0.000198 throughput=263.6/s elapsed=8s eta=2m08s
[LIVE] 2/20 batch=6/6 loss=1.6224 lr=0.000195 throughput=274.0/s elapsed=10s eta=1m33s
[VALID] 2/20 val_loss=1.7366 val_acc=0.1824 macro_f1=0.1126 bal_acc=0.3500 gap=0.1297
[BEST] 2/20 val_loss=1.7366
[CKPT] epoch_end epoch=2 step=12
[LIVE] 3/20 batch=1/6 loss=1.6102 lr=0.000194 throughput=267.0/s elapsed=18s eta=2m32s
[LIVE] 3/20 batch=6/6 loss=1.6000 lr=0.000189 throughput=274.4/s elapsed=21s eta=1m58s
[VALID] 3/20
[LIVE] 4/20 batch=1/6 loss=1.6048 lr=0.000188 throughput=273.3/s elapsed=23s eta=2m02s
[LIVE] 4/20 batch=6/6 loss=1.5954 lr=0.000181 throughput=274.4/s elapsed=25s eta=1m41s
[VALID] 4/20 val_loss=1.5467 val_acc=0.2973 macro_f1=0.1261 bal_acc=0.2238 gap=-0.0539
[BEST] 4/20 val_loss=1.5467
[CKPT] epoch_end epoch=4 step=24
[LIVE] 5/20 batch=1/6 loss=1.5989 lr=0.000179 throughput=266.0/s elapsed=33s eta=2m07s
[LIVE] 5/20 batch=6/6 loss=1.5878 lr=0.000171 throughput=273.8/s elapsed=36s eta=1m4

In [20]:
#@title OOD Calibration + Adapter Save
with TELEMETRY.capture_cell_output("Cell 7: OOD Calibration + Adapter Save"):
    from scripts.colab_notebook_helpers import calibrate_and_save_notebook_adapter

    result = calibrate_and_save_notebook_adapter(
        root=ROOT,
        state=STATE,
        telemetry=TELEMETRY,
        rt_fn=rt,
        print_fn=print,
    )
    STATE["calibration"] = result["calibration"]
    STATE["adapter_export_dir"] = result["adapter_export_dir"]


[OOD] Kalibrasyon tamamlandi. classes=5 version=1
Cell 7: calibration and adapter save started
Kaydedilen adapter klasoru:
/content/bitirmeprojesi/outputs/colab_notebook_training/continual_sd_lora_adapter
 - outputs/colab_notebook_training/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/grape_fruit_2026-04-14_10-56-34/artifacts/adapter_export/continual_sd_lora_adapter
Cell 7: calibration and adapter save completed


In [ ]:
#@title Final Evaluation + Run Finalization
with TELEMETRY.capture_cell_output("Cell 8: Final Evaluation + Run Finalization"):
    from scripts.colab_notebook_helpers import complete_notebook_training_run

    completion_result = complete_notebook_training_run(
        root=ROOT,
        state=STATE,
        base_config=BASE_CONFIG,
        crop_name=CROP_NAME,
        part_name=PART_NAME,
        run_id=RUN_ID,
        device=DEVICE,
        epochs=EPOCHS,
        runtime_dataset_root=RUNTIME_DATASET_ROOT,
        repo_run_dir=REPO_RUN_DIR,
        repo_output_dir=REPO_OUTPUT_DIR,
        repo_telemetry_dir=REPO_TELEMETRY_DIR,
        repo_checkpoint_state_dir=REPO_CHECKPOINT_STATE_DIR,
        repo_notebook_output_path=REPO_NOTEBOOK_OUTPUT_PATH,
        auto_push_to_github=AUTO_PUSH_TO_GITHUB,
        auto_push_remote_name=AUTO_PUSH_REMOTE_NAME,
        auto_push_branch=AUTO_PUSH_BRANCH,
        auto_disconnect_runtime=AUTO_DISCONNECT_RUNTIME,
        auto_disconnect_grace_seconds=AUTO_DISCONNECT_GRACE_SECONDS,
        save_run_outputs_to_repo_fn=save_run_outputs_to_repo,
        export_current_colab_notebook_fn=export_current_colab_notebook,
        push_repo_run_to_github_fn=push_repo_run_to_github,
        telemetry=TELEMETRY,
        print_fn=print,
    )
    STATE.update(completion_result["state"])


[DOGRULAMA (referans)] ornek=148 sinif=5 accuracy=0.7027 ood_auroc=0.5433 sure_ds_f1=0.0972 conformal_cov=0.9595
[TEST (ayrilmis)] ornek=148 sinif=5 accuracy=0.7230 ood_auroc=0.6052 sure_ds_f1=0.0848 conformal_cov=0.9865
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.


In [ ]:
#@title Advanced: Secure Sanitize + Repo Push
with TELEMETRY.capture_cell_output("Advanced: Secure Sanitize + Repo Push"):

    import json

    import re

    import subprocess

    from pathlib import Path



    RUN_ID_TO_PUSH = str(globals().get("RUN_ID_TO_PUSH", "") or RUN_ID).strip()

    if not RUN_ID_TO_PUSH:

        raise RuntimeError("RUN_ID_TO_PUSH cozulmedi.")



    run_dir = ROOT / "runs" / RUN_ID_TO_PUSH

    if not run_dir.is_dir():

        raise RuntimeError(f"Run klasoru bulunamadi: {run_dir}")



    gh_token = str(resolve_github_token() or "").strip()

    hf_token = str(resolve_hf_token() or "").strip()

    if not gh_token:

        raise RuntimeError("GitHub token bulunamadi (GH_TOKEN/GITHUB_TOKEN gerekli).")



    MAX_FILE_SIZE_BYTES = 95 * 1024 * 1024

    EXCLUDED_SUFFIXES = {".pt"}

    EXCLUDED_PATH_PARTS = {"checkpoint_state/checkpoints"}



    token_patterns = [

        re.compile(r"\bgh[pousr]_[A-Za-z0-9_]{10,}\b"),

        re.compile(r"\bgithub_pat_[A-Za-z0-9_]{10,}\b"),

        re.compile(r"https://[^\s:@]+:[^\s@]+@github\.com", re.IGNORECASE),

    ]

    explicit_tokens = [token for token in (gh_token, hf_token) if token]



    def _redact_text(value: str) -> str:

        text = str(value or "")

        for secret in explicit_tokens:

            text = text.replace(secret, "<redacted>")

        for pattern in token_patterns:

            text = pattern.sub("<redacted>", text)

        return text



    def _redact_obj(obj):

        if isinstance(obj, dict):

            return {k: _redact_obj(v) for k, v in obj.items()}

        if isinstance(obj, list):

            return [_redact_obj(item) for item in obj]

        if isinstance(obj, str):

            return _redact_text(obj)

        return obj



    def _git(args, check=True):

        completed = subprocess.run(

            ["git", *args],

            cwd=str(ROOT),

            text=True,

            capture_output=True,

            check=False,

        )

        if check and completed.returncode != 0:

            raise RuntimeError((completed.stderr or completed.stdout or "git command failed").strip())

        return completed



    def _git_push_with_header(remote_name: str, branch_name: str):

        completed = subprocess.run(

            [

                "git",

                "-c",

                f"http.extraheader=AUTHORIZATION: bearer {gh_token}",

                "push",

                remote_name,

                f"HEAD:{branch_name}",

            ],

            cwd=str(ROOT),

            text=True,

            capture_output=True,

            check=False,

        )

        if completed.returncode != 0:

            output = (completed.stdout or "") + "\n" + (completed.stderr or "")

            raise RuntimeError(f"Push basarisiz.\n{output.strip()}")

        return completed



    def _is_excluded(path: Path) -> bool:

        rel = path.relative_to(run_dir).as_posix().lower()

        if path.suffix.lower() in EXCLUDED_SUFFIXES:

            return True

        if any(part in rel for part in EXCLUDED_PATH_PARTS):

            return True

        try:

            if path.stat().st_size > MAX_FILE_SIZE_BYTES:

                return True

        except OSError:

            return True

        return False



    # 1) Run klasorundeki text tabanli artefaktlari sanitize et

    scanned = 0

    redacted_files = 0

    leaks = []

    for path in sorted(run_dir.rglob("*")):

        if not path.is_file():

            continue

        suffix = path.suffix.lower()

        if suffix not in {".json", ".jsonl", ".log", ".md", ".txt"}:

            continue



        scanned += 1

        raw = path.read_text(encoding="utf-8", errors="ignore")

        if suffix == ".json":

            try:

                obj = json.loads(raw)

                new_raw = json.dumps(_redact_obj(obj), ensure_ascii=False, indent=2) + "\n"

            except Exception:

                new_raw = _redact_text(raw)

        elif suffix == ".jsonl":

            lines = []

            for line in raw.splitlines():

                stripped = line.strip()

                if not stripped:

                    lines.append(line)

                    continue

                try:

                    obj = json.loads(stripped)

                    lines.append(json.dumps(_redact_obj(obj), ensure_ascii=False))

                except Exception:

                    lines.append(_redact_text(line))

            new_raw = "\n".join(lines) + ("\n" if raw.endswith("\n") else "")

        else:

            new_raw = _redact_text(raw)



        if new_raw != raw:

            path.write_text(new_raw, encoding="utf-8")

            redacted_files += 1

        if any(p.search(new_raw) for p in token_patterns):

            leaks.append(path.as_posix())



    print(f"[SECURITY] scanned={scanned} redacted_files={redacted_files}")

    if leaks:

        raise RuntimeError(f"Push oncesi hala secret pattern var: {leaks[:10]}")



    # 2) Local branch'i remote ile hizala (reddedilen commit zincirinden temiz basla)

    remote_name = str(globals().get("AUTO_PUSH_REMOTE_NAME", "origin") or "origin").strip()

    current_branch = _git(["rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()

    target_branch = str(globals().get("AUTO_PUSH_BRANCH", "") or current_branch).strip()

    if not target_branch:

        raise RuntimeError("Hedef branch cozulmedi.")



    _git(["fetch", remote_name, target_branch])

    _git(["reset", "--soft", f"{remote_name}/{target_branch}"])

    _git(["reset"])



    # 3) Sadece uygun dosyalari stage et (.pt, checkpoint blobs, ve buyuk dosyalar haric)

    relative_run = f"runs/{RUN_ID_TO_PUSH}"

    tracked_files = []

    skipped_files = []

    for path in sorted(run_dir.rglob("*")):

        if not path.is_file():

            continue

        rel_to_repo = path.relative_to(ROOT).as_posix()

        if _is_excluded(path):

            skipped_files.append(rel_to_repo)

            continue

        tracked_files.append(rel_to_repo)



    for file_path in tracked_files:

        _git(["add", "-f", "--", file_path])



    # Daha once index'e girmis olabilecek buyuk/checkpoint dosyalari garantili temizle

    if skipped_files:

        _git(["rm", "--cached", "-r", "--ignore-unmatch", "--", *skipped_files], check=False)



    staged = _git(["diff", "--cached", "--name-only"], check=False).stdout.strip().splitlines()

    staged = [line for line in staged if line.strip().startswith(relative_run + "/")]



    print(f"[GIT] stage hazir: staged={len(staged)} skipped={len(skipped_files)}")

    if skipped_files:

        print("[GIT] skip examples:", skipped_files[:5])



    if not staged:

        report = {

            "enabled": True,

            "pushed": False,

            "run_dir": str(run_dir),

            "message": "Staged degisiklik yok; pushlanacak uygun dosya bulunamadi.",

            "branch": target_branch,

            "skipped_files": len(skipped_files),

        }

        STATE["git_push_report"] = report

        print("[GIT] secure push report:", report)

    else:

        commit_message = f"Add notebook 2 outputs for run {RUN_ID_TO_PUSH} (safe artifacts only)"

        _git(["commit", "-m", commit_message])

        _git_push_with_header(remote_name, target_branch)

        report = {

            "enabled": True,

            "pushed": True,

            "run_dir": str(run_dir),

            "branch": target_branch,

            "staged_files": len(staged),

            "skipped_files": len(skipped_files),

        }

        STATE["git_push_report"] = report

        print("[GIT] secure push report:", report)
